# Tabular XGBoost Experimentation & Tuning
**Objective:** Engineer tabular metadata, build a robust scikit-learn preprocessing pipeline, and find the optimal XGBoost hyperparameters using GridSearchCV.

**Professional Standards Implemented:**
* Complete pipeline integration (Imputation + Encoding) to prevent data leakage.
* GridSearchCV for hyperparameter tuning.
* Robust missing-value handling for inference time.

## Imports and Setup

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xbg
import shap

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## Data Loading and Merging

### Loading Data

In [2]:
targets_df= pd.read_csv(filepath_or_buffer= os.path.abspath('../data/processed/time_series_targets.csv'),
                        dtype= {'article_id': str})

cnn_scores_df= pd.read_csv(filepath_or_buffer= os.path.abspath('../data/processed/cnn_visual_scores.csv'),
                        dtype= {'article_id': str})

articles_df= pd.read_csv(filepath_or_buffer= os.path.abspath('../data/raw/articles.csv'),
                        dtype= {'article_id': str})

In [3]:
targets_df.head()

,article_id,month,monthly_sales
0,0108775015,2018-09,662
1,0108775015,2018-10,1532
2,0108775015,2018-11,1660
3,0108775015,2018-12,1207
4,0108775015,2019-01,1522


In [4]:
targets_df.shape

(768883, 3)

In [5]:
cnn_scores_df.head()

,article_id,CNN_Visual_Trend_Score
0,0111593001,0.305146
1,0112679048,0.236609
2,0114428030,0.389910
3,0118458004,0.275760
4,0118458028,0.230257


In [6]:
cnn_scores_df.shape

(9961, 2)

In [7]:
articles_df.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [8]:
articles_df.shape

(105542, 25)

### Merging Data

#### First Merge

In [9]:
df= pd.merge(left= targets_df,
             right= cnn_scores_df,
             on= 'article_id',
             how= 'inner')

In [10]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score
0,0111593001,2018-09,530,0.305146
1,0111593001,2018-10,1207,0.305146
2,0111593001,2018-11,1027,0.305146
3,0111593001,2018-12,1558,0.305146
4,0111593001,2019-01,504,0.305146


In [11]:
df.shape

(73291, 4)

#### Final Merge

In [12]:
articles_df.columns

Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')

In [13]:
articles_df.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [14]:
columns_to_merge= [
    'article_id', 'prod_name', 'product_type_name', 'product_group_name',
    'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name',
    'perceived_colour_master_name', 'department_name', 'index_name',
    'index_group_name', 'section_name', 'garment_group_name'
]

In [15]:
# Final Merge:
df = pd.merge(left= df,
              right= articles_df[columns_to_merge],
              on= 'article_id',
              how= 'inner')

In [16]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name
0,0111593001,2018-09,530,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
1,0111593001,2018-10,1207,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
2,0111593001,2018-11,1027,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
3,0111593001,2018-12,1558,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
4,0111593001,2019-01,504,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights


In [17]:
df.shape

(73291, 16)

In [18]:
# Adding Month as a Separate Column:
df['month_int'] = pd.to_datetime(df['month']).dt.month

In [19]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,month_int
0,0111593001,2018-09,530,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,9
1,0111593001,2018-10,1207,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,10
2,0111593001,2018-11,1027,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,11
3,0111593001,2018-12,1558,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,12
4,0111593001,2019-01,504,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,1


In [20]:
df.shape

(73291, 17)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73291 entries, 0 to 73290
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   article_id                    73291 non-null  object 
 1   month                         73291 non-null  object 
 2   monthly_sales                 73291 non-null  int64  
 3   CNN_Visual_Trend_Score        73291 non-null  float64
 4   prod_name                     73291 non-null  object 
 5   product_type_name             73291 non-null  object 
 6   product_group_name            73291 non-null  object 
 7   graphical_appearance_name     73291 non-null  object 
 8   colour_group_name             73291 non-null  object 
 9   perceived_colour_value_name   73291 non-null  object 
 10  perceived_colour_master_name  73291 non-null  object 
 11  department_name               73291 non-null  object 
 12  index_name                    73291 non-null  object 
 13  i

In [ ]:
# Checking Uniques in all Object Type Columns to address possible Dimensionality Issue:
for col in df.select_dtypes(include= 'object'):
    print(f"{col} - Uniques: {df[col].nunique()}\n")

article_id - Uniques: 9961

month - Uniques: 25

prod_name - Uniques: 8457

product_type_name - Uniques: 99

product_group_name - Uniques: 15

graphical_appearance_name - Uniques: 30

colour_group_name - Uniques: 50

perceived_colour_value_name - Uniques: 8

perceived_colour_master_name - Uniques: 19

department_name - Uniques: 235

index_name - Uniques: 10

index_group_name - Uniques: 5

section_name - Uniques: 55

garment_group_name - Uniques: 21



In [23]:
# Dropping 'prod_name' column for having too many categories, working essentially as unique:
df= df.drop('prod_name', axis= 1)

In [24]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,month_int
0,0111593001,2018-09,530,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,9
1,0111593001,2018-10,1207,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,10
2,0111593001,2018-11,1027,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,11
3,0111593001,2018-12,1558,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,12
4,0111593001,2019-01,504,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,1


In [25]:
# Checking Uniques in all Object Type Columns to address possible Dimensionality Issue:
for col in df.select_dtypes(include= 'object'):
    print(f"{col} - Uniques: {df[col].nunique()}\n")

article_id - Uniques: 9961

month - Uniques: 25

product_type_name - Uniques: 99

product_group_name - Uniques: 15

graphical_appearance_name - Uniques: 30

colour_group_name - Uniques: 50

perceived_colour_value_name - Uniques: 8

perceived_colour_master_name - Uniques: 19

department_name - Uniques: 235

index_name - Uniques: 10

index_group_name - Uniques: 5

section_name - Uniques: 55

garment_group_name - Uniques: 21

